# Stage 1 — Classical baselines

**Goal:** build the reference point that every neural model in later stages has to beat.

**What this notebook does**
1. Loads the Stage 0 outputs (`splits.csv`, tracked landmarks).
2. Turns every clip into **one row of 648 numbers** (shape, motion and position of the hands).
3. Trains classical models from scikit-learn: majority class, kNN, PCA + logistic regression, linear SVM, random forest.
4. Evaluates them on **signers the model has never seen** (validation and test folds), then rotates over all 5 folds.
5. Runs three small experiments: which feature groups matter, whether down-weighting prolific signers helps, and how much **signer leakage** inflates accuracy on the official test set.

**How to run:** CPU runtime, cells in order, starting at Cell 1. Rough times on Colab CPU (estimates, not measured): feature extraction 1–3 min, fast models seconds each, linear SVM 1–3 min, random forest 3–8 min.
Set `RUN_SLOW_MODELS = False` in Cell 1 to skip the SVM and the forest while you read the results.

**Reminder of the reading rule:** a number only means something next to its baseline. Chance level is 1 / (number of signs).

In [ ]:
# [setup] Cell 1 — configuration
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
FIXED_T = 64             # frames per clip after resampling (same as Stage 0)
N_SEGMENTS = 3           # the clip is cut into 3 time segments so the features keep a rough notion of order
PCA_DIM = 128            # dimensions kept by PCA
SUBSET_ONLY = True       # True: the development subset from Stage 0; False: all 1,000 signs (slower)
RUN_SLOW_MODELS = True   # linear SVM + random forest in the single-split comparison (SVM ~10 min, forest ~2 min per fit)
CV_INCLUDE_SVM = False   # the SVM in 5-fold CV took over an hour on Colab; kNN and PCA + logistic regression take minutes
N_FOLDS = 5
warnings.filterwarnings('ignore')
print('Working directory:', BASE)

In [ ]:
# Cell 2 — load the Stage 0 outputs
splits = pd.read_csv(PROC / 'splits.csv')
z = np.load(PROC / 'landmarks_raw.npz', allow_pickle=False)
ids, lengths = z['ids'], z['lengths']
offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
frames = np.load(PROC / 'frames_tracked_v2.npy', mmap_mode='r')      # hands with stable slots (Stage 0, cell 11b)

assert len(frames) == offsets[-1], 'frames_tracked_v2.npy does not match landmarks_raw.npz: re-run Stage 0 cells 10 and 11b.'
assert splits['attachment_id'].isin(list(id2idx)).all(), 'some clips in splits.csv have no landmarks'

meta = (splits[splits['in_subset']] if SUBSET_ONLY else splits).reset_index(drop=True)
LABEL = 'label_id_subset' if SUBSET_ONLY else 'label_id_full'
y_all = meta[LABEL].astype(int).to_numpy()
N_CLASSES = int(len(np.unique(y_all)))
print(f'{len(meta)} clips | {N_CLASSES} signs | chance level {1 / N_CLASSES:.2%}')
print(meta['split'].value_counts().to_dict(), '| signers:', meta['user_id'].nunique())

In [ ]:
# [features] Cell 3 — from a clip to one row of numbers
WRIST, MIDDLE_MCP = 0, 9

def normalize_hands(x, aspect=None, eps=1e-6):
    """x: (..., 21, 3). Correct x for the video aspect ratio, centre on the wrist, divide by the
    wrist->middle-finger-base distance. NaN (missing hand) stays NaN."""
    x = x.astype(np.float32).copy()
    if aspect is not None:
        x[..., 0] *= aspect
    centred = x - x[..., WRIST:WRIST + 1, :]
    scale = np.linalg.norm(centred[..., MIDDLE_MCP, :2], axis=-1)
    scale = np.where(scale < eps, np.nan, scale)
    return centred / scale[..., None, None]

def clip_features(clip, aspect, T=FIXED_T, n_seg=N_SEGMENTS):
    """One clip (L, 2, 21, 3) -> three feature groups.
    shape:    per hand, mean of the normalized joints in each time segment + their spread over the clip
    motion:   per hand, mean absolute change of the normalized joints between consecutive frames
    position: per hand, where the wrist is in the image (mean, spread, min, max) + how often the hand is visible"""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    hands = normalize_hands(clip, aspect=aspect)[idx]                      # (T, 2, 21, 3)
    wrist = clip[:, :, WRIST, :2].astype(np.float32).copy()
    wrist[..., 0] *= aspect
    wrist = wrist[idx]                                                     # (T, 2, 2)
    present = ~np.isnan(hands).all(axis=(2, 3))                            # (T, 2)
    segs = np.array_split(np.arange(T), n_seg)
    shape_f, motion_f, pos_f, pres_f = [], [], [], []
    for h in range(2):
        ok = present[:, h]
        if ok.sum() == 0:                                                  # hand never visible: zeros
            shape_f.append(np.zeros(n_seg * 63 + 63)); motion_f.append(np.zeros(63))
            pos_f.append(np.zeros(8)); pres_f.append(0.0)
            continue
        Hh = hands[:, h].reshape(T, 63)
        overall = Hh[ok].mean(0)
        seg_means = [Hh[s][ok[s]].mean(0) if ok[s].any() else overall for s in segs]
        shape_f.append(np.concatenate(seg_means + [Hh[ok].std(0)]))
        Hp = Hh[ok]
        motion_f.append(np.abs(np.diff(Hp, axis=0)).mean(0) if len(Hp) > 1 else np.zeros(63))
        Wp = wrist[:, h][ok]
        pos_f.append(np.concatenate([Wp.mean(0), Wp.std(0), Wp.min(0), Wp.max(0)]))
        pres_f.append(float(ok.mean()))
    return (np.concatenate(shape_f), np.concatenate(motion_f), np.concatenate(pos_f + [np.array(pres_f)]))

# --- probe the sizes of the three groups and run unit checks on synthetic data ---
_rng = np.random.default_rng(0)
_clip = _rng.normal(size=(40, 2, 21, 3)).astype(np.float32)
_s, _m, _p = clip_features(_clip, 1.0)
N_SHAPE, N_MOTION, N_POS = len(_s), len(_m), len(_p)
SHAPE, MOTION, POS = (slice(0, N_SHAPE), slice(N_SHAPE, N_SHAPE + N_MOTION),
                      slice(N_SHAPE + N_MOTION, N_SHAPE + N_MOTION + N_POS))
N_FEATURES = N_SHAPE + N_MOTION + N_POS

_moved = _clip * 2.0 + np.array([5.0, -3.0, 0.0], np.float32)             # same hands: bigger and shifted
_s2, _m2, _p2 = clip_features(_moved, 1.0)
assert np.allclose(_s, _s2, atol=1e-4) and np.allclose(_m, _m2, atol=1e-4), 'shape/motion must not depend on position or size'
assert not np.allclose(_p, _p2), 'position features must change when the hand moves'
_gone = _clip.copy(); _gone[:, 1] = np.nan                                  # second hand never visible
assert np.isfinite(np.concatenate(clip_features(_gone, 1.0))).all()
print(f'OK: unit checks passed | features: shape {N_SHAPE} + motion {N_MOTION} + position {N_POS} = {N_FEATURES}')

In [ ]:
# Cell 4 — compute the features for every clip (cached on Drive)
from tqdm.auto import tqdm

FEAT = PROC / f'features_stage1_{"subset" if SUBSET_ONLY else "all"}_T{FIXED_T}_S{N_SEGMENTS}.npz'
meta_ids = meta['attachment_id'].to_numpy().astype(str)

X = None
if FEAT.exists():
    zf = np.load(FEAT, allow_pickle=False)
    if zf['ids'].shape == meta_ids.shape and (zf['ids'] == meta_ids).all() and zf['X'].shape[1] == N_FEATURES:
        X = zf['X']
        print('Loaded cached features:', FEAT.name)
if X is None:
    X = np.zeros((len(meta), N_FEATURES), np.float32)
    for r, (cid, asp) in enumerate(tqdm(zip(meta['attachment_id'], meta['aspect']), total=len(meta))):
        i = id2idx[cid]
        clip = np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32)
        X[r] = np.concatenate(clip_features(clip, float(asp)))
    np.savez_compressed(FEAT, X=X, ids=meta_ids)
    print('Saved', FEAT.name)

assert np.isfinite(X).all()
print('feature matrix:', X.shape, X.dtype)

In [ ]:
# Cell 5 — sanity checks on the feature matrix
presence = X[:, N_SHAPE + N_MOTION + N_POS - 2:]                            # last two columns = visibility of hand 0 / hand 1
print(f'clips where slot 1 (second hand) is visible in more than half of the frames: {(presence[:, 1] > 0.5).mean():.1%}')
print(f'clips where slot 0 is never visible: {(presence[:, 0] == 0).mean():.2%}')
std = X.std(0)
print(f'constant features: {(std < 1e-8).sum()} of {N_FEATURES}')

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, (name, sl) in zip(ax, (('shape', SHAPE), ('motion', MOTION), ('position', POS))):
    a.hist(X[:, sl].ravel(), bins=60, color='#4C78A8'); a.set_title(f'{name} features (all values)'); a.set_yscale('log')
plt.tight_layout(); plt.show()

In [ ]:
# [models] Cell 6 — helpers: models, metrics, training
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

def make_model(name, hp=None, seed=SEED):
    hp = hp or {}
    if name == 'majority class':
        return DummyClassifier(strategy='most_frequent')
    if name == 'kNN':
        return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=hp.get('k', 5), metric='cosine', algorithm='brute'))
    if name == 'PCA + logistic regression':
        return make_pipeline(StandardScaler(), PCA(n_components=PCA_DIM, random_state=seed),
                             LogisticRegression(C=hp.get('C', 0.1), max_iter=300))
    if name == 'linear SVM':
        return make_pipeline(StandardScaler(), LinearSVC(C=hp.get('C', 0.05), max_iter=500))
    if name == 'random forest':
        return RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=seed)
    raise ValueError(name)

def topk_accuracy(scores, y_true, classes, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return float(np.mean([yt in classes[t] for yt, t in zip(y_true, top)]))

def train(model, Xtr, ytr, sample_weight=None):
    t = time.time()
    if sample_weight is None:
        model.fit(Xtr, ytr)
    else:
        model.fit(Xtr, ytr, **{f'{model.steps[-1][0]}__sample_weight': sample_weight})
    return time.time() - t

def scores_of(model, Xe):
    return model.predict_proba(Xe) if hasattr(model, 'predict_proba') else model.decision_function(Xe)

def evaluate(model, Xe, ye):
    sc = scores_of(model, Xe)
    pred = model.classes_[sc.argmax(1)]
    return {'n': int(len(ye)), 'top1': float((pred == ye).mean()), 'top5': topk_accuracy(sc, ye, model.classes_, 5),
            'macro_f1': float(f1_score(ye, pred, average='macro')), 'pred': pred}

results = []          # one dict per (experiment, model, evaluation set)
def log(experiment, model_name, set_name, m, fit_s=None, **extra):
    results.append({'experiment': experiment, 'model': model_name, 'set': set_name, 'n': m['n'],
                    'top1': m['top1'], 'top5': m['top5'], 'macro_f1': m['macro_f1'], 'fit_s': fit_s, **extra})

tr = (meta['split'] == 'train').to_numpy(); va = (meta['split'] == 'val').to_numpy(); te = (meta['split'] == 'test').to_numpy()
print(f'train {tr.sum()} | val {va.sum()} | test {te.sum()} clips')

In [ ]:
# Cell 7 — baseline reference: what does "no model" score?
m = make_model('majority class'); fs = train(m, X[tr], y_all[tr])
for name, mask in (('val', va), ('test', te)):
    r = evaluate(m, X[mask], y_all[mask]); log('single split', 'majority class', name, r, fs)
    print(f'majority class on {name}: top-1 {r["top1"]:.2%} (chance level {1 / N_CLASSES:.2%})')

In [ ]:
# Cell 8 — tune the one hyper-parameter of each fast model on the VALIDATION fold (unseen signers)
GRID = {'kNN': [{'k': k} for k in (1, 3, 5, 10)],
        'PCA + logistic regression': [{'C': c} for c in (0.003, 0.03, 0.3, 3.0)]}
BEST = {'linear SVM': {'C': 0.05}, 'random forest': {}}

for name, grid in GRID.items():
    rows = []
    for hp in grid:
        mdl = make_model(name, hp); train(mdl, X[tr], y_all[tr])
        rows.append((evaluate(mdl, X[va], y_all[va])['top1'], hp))
    best = max(rows, key=lambda r: r[0])
    BEST[name] = best[1]
    print(f'{name}: ' + ' | '.join(f'{hp} -> {acc:.1%}' for acc, hp in rows) + f'   => chosen {best[1]}')

In [ ]:
# Cell 9 — main comparison on the signer-disjoint split (train on train, report val and test)
MODELS = ['kNN', 'PCA + logistic regression'] + (['linear SVM', 'random forest'] if RUN_SLOW_MODELS else [])
for name in MODELS:
    mdl = make_model(name, BEST.get(name)); fs = train(mdl, X[tr], y_all[tr])
    line = []
    for sname, mask in (('val', va), ('test', te)):
        r = evaluate(mdl, X[mask], y_all[mask]); log('single split', name, sname, r, fs, hp=str(BEST.get(name)))
        line.append(f'{sname} top-1 {r["top1"]:.1%} top-5 {r["top5"]:.1%}')
    print(f'{name:28s} fit {fs:5.1f}s | ' + ' | '.join(line))

In [ ]:
# Cell 10 — experiment A: which feature groups matter? (PCA + logistic regression, validation and test)
groups = {'shape only': [SHAPE], 'shape + motion': [SHAPE, MOTION], 'shape + position': [SHAPE, POS],
          'position only': [POS], 'all features': [SHAPE, MOTION, POS]}
def cols(parts):
    return np.concatenate([np.arange(N_FEATURES)[s] for s in parts])

for gname, parts in groups.items():
    c = cols(parts)
    mdl = make_model('PCA + logistic regression', BEST['PCA + logistic regression'])
    mdl.set_params(pca__n_components=min(PCA_DIM, len(c)))
    fs = train(mdl, X[tr][:, c], y_all[tr])
    out = []
    for sname, mask in (('val', va), ('test', te)):
        r = evaluate(mdl, X[mask][:, c], y_all[mask]); log('feature groups', gname, sname, r, fs); out.append(f'{sname} {r["top1"]:.1%}')
    print(f'{gname:18s} ({len(c):3d} features) top-1: ' + ' | '.join(out))

In [ ]:
# Cell 11 — experiment B: seven prolific signers dominate training. Does down-weighting them help unseen signers?
def signer_weights(users):
    s = pd.Series(users)
    w = 1.0 / np.sqrt(s.map(s.value_counts()).to_numpy())
    return w / w.mean()

for label, w in (('equal weight per clip', None), ('weight ~ 1/sqrt(signer clips)', signer_weights(meta.loc[tr, 'user_id'].to_numpy()))):
    mdl = make_model('PCA + logistic regression', BEST['PCA + logistic regression'])
    fs = train(mdl, X[tr], y_all[tr], sample_weight=w)
    out = []
    for sname, mask in (('val', va), ('test', te)):
        r = evaluate(mdl, X[mask], y_all[mask]); log('signer weighting', label, sname, r, fs); out.append(f'{sname} {r["top1"]:.1%}')
    print(f'{label:32s} top-1: ' + ' | '.join(out))

In [ ]:
# Cell 12 — five-fold cross-validation over signers (every non-prolific clip is tested exactly once)
folds = meta['fold'].to_numpy()          # -1 = prolific signers: always in train
cv_models = ['kNN', 'PCA + logistic regression'] + (['linear SVM'] if (RUN_SLOW_MODELS and CV_INCLUDE_SVM) else [])
cv_acc = {m: [] for m in cv_models}
pooled = {m: ([], []) for m in cv_models}
for k in range(N_FOLDS):
    te_k, tr_k = folds == k, folds != k
    for name in cv_models:
        mdl = make_model(name, BEST.get(name)); fs = train(mdl, X[tr_k], y_all[tr_k])
        r = evaluate(mdl, X[te_k], y_all[te_k]); log('5-fold CV', name, f'fold {k}', r, fs)
        cv_acc[name].append(r['top1']); pooled[name][0].append(y_all[te_k]); pooled[name][1].append(r['pred'])
    print(f'fold {k}: test clips {te_k.sum()} | ' + ' | '.join(f'{m} {cv_acc[m][-1]:.1%}' for m in cv_models))
print()
for name in cv_models:
    yt, yp = np.concatenate(pooled[name][0]), np.concatenate(pooled[name][1])
    print(f'{name:28s} top-1 mean {np.mean(cv_acc[name]):.1%} +/- {np.std(cv_acc[name]):.1%} (over folds) | pooled {np.mean(yt == yp):.1%} on {len(yt)} clips')

In [ ]:
# Cell 13 — experiment C: how much does signer leakage inflate accuracy? (official split)
off_train = meta['official_train'].to_numpy(bool)
unseen = meta['official_test_unseen_signer'].to_numpy(bool)
seen = (~off_train) & (~unseen)
print(f'official train {off_train.sum()} clips | official test: {seen.sum()} from signers seen in training, {unseen.sum()} from unseen signers')

for name in ['kNN', 'PCA + logistic regression']:
    mdl = make_model(name, BEST.get(name)); fs = train(mdl, X[off_train], y_all[off_train])
    out = {}
    for sname, mask in (('official test: seen signers', seen), ('official test: unseen signers', unseen)):
        r = evaluate(mdl, X[mask], y_all[mask]); log('leakage', name, sname, r, fs); out[sname] = r['top1']
    gap = out['official test: seen signers'] - out['official test: unseen signers']
    print(f'{name:28s} seen {out["official test: seen signers"]:.1%} | unseen {out["official test: unseen signers"]:.1%} | gap {gap:+.1%}')

In [ ]:
# Cell 14 — results table and figure
res = pd.DataFrame(results)
res.to_csv(RES / 'stage1_results.csv', index=False)

main = res[(res.experiment == 'single split') & (res.set == 'test')].set_index('model')
chance = 1 / N_CLASSES
print(f'chance level: {chance:.2%}')
print(main[['top1', 'top5', 'macro_f1', 'fit_s']].round(3))
print('Note: top-5 is not meaningful for kNN with k=1 (it only scores one class).')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
mm = res[res.experiment == 'single split']
order = [m for m in ['majority class', 'kNN', 'PCA + logistic regression', 'linear SVM', 'random forest'] if m in set(mm.model)]
xx = np.arange(len(order)); w = 0.38
for j, (sname, col) in enumerate((('val', '#F58518'), ('test', '#54A24B'))):
    ax[0].bar(xx + (j - .5) * w, [mm[(mm.model == m) & (mm.set == sname)].top1.iloc[0] for m in order], w, label=sname, color=col)
ax[0].axhline(chance, color='k', ls='--', lw=1, label='chance'); ax[0].set_xticks(xx); ax[0].set_xticklabels(order, rotation=20, ha='right')
ax[0].set_ylabel('top-1 accuracy'); ax[0].set_title('Signer-disjoint split'); ax[0].legend()
lk = res[res.experiment == 'leakage']
if len(lk):
    for j, (sname, col) in enumerate((('official test: seen signers', '#B279A2'), ('official test: unseen signers', '#4C78A8'))):
        names = list(lk.model.unique())
        ax[1].bar(np.arange(len(names)) + (j - .5) * w, [lk[(lk.model == m) & (lk.set == sname)].top1.iloc[0] for m in names], w,
                  label=sname.replace('official test: ', ''), color=col)
    ax[1].set_xticks(np.arange(len(names))); ax[1].set_xticklabels(names); ax[1].set_title('Signer leakage: seen vs unseen signers'); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / 'stage1_results.png', dpi=150); plt.show()
print('Saved', RES / 'stage1_results.csv', 'and', FIG / 'stage1_results.png')

## Stage 1 checklist

- [ ] Cell 3 printed `OK: unit checks passed`
- [ ] Cell 7: the majority-class baseline is close to chance
- [ ] Cell 9: at least one model clearly beats chance on both val and test
- [ ] Cell 13: seen vs unseen accuracy for the leakage experiment
- [ ] `results/stage1_results.csv` and `figures/stage1_results.png` exist

**Send back:** the printed outputs of Cells 8, 9, 10, 11, 12 and 13, the table from Cell 14, and the figure.
With these I will write the Stage 1 results into the report and prepare Stage 2 (the first neural network, an MLP).